# Solar Filament Segmentation Challenge 2026
## Apex Sovereign: End-to-End Scientific Pipeline Walkthrough (Global Rank 1 — 0.64 PQ)
**Team:** Tempest (`tensura911`) | **Competition:** Solar Filament Segmentation Challenge 2026 (ESAIR Lab / NSF / NSO)

---

### Architectural Overview
This self-contained notebook demonstrates and executes the complete **Apex Sovereign** machine learning and morphological calibration framework:
1. **Solar Disk Limb Detection & Radial Preprocessing (`src.preprocessing`)**: Automated Otsu limb circle estimation, quadratic radial limb darkening correction (LDC), robust on-disk Z-score normalization, and positive-biased ($75\%$ foreground-centered) crop sampling across 5 Carrington-rotation-grouped folds.
2. **Multi-Backbone Deep Neural Ensemble (`src.models`)**: Combines spatial residual representations (**ResNet-34 U-Net**), multi-scale depthwise separable features (**EfficientNet-B3 U-Net** under Soft Dice and Asymmetric Tversky $\alpha=0.3, \beta=0.7$ losses), and dense feature reuse (**DenseNet-121 U-Net** under submodular Lovász hinge loss), evaluated with 8-way dihedral ($D_4$) test-time augmentation in uncompressed log-odds (logit) space.
3. **Decoupled Topological Spine Grouping ($R=6$) & 18-Feature GBDT Filter (`src.postprocess`)**: Bridges 3–6 pixel chromospheric absorption dips using an elliptical structuring element $K_R$ ($13\times 13$ px) at $T_{\text{group}}=0.40$ while preserving crisp boundaries at $T_{\text{trim}}=0.480$, followed by an 18-feature Histogram GBDT classifier ($\tau=0.38$) that prunes non-filament sunspot pores and chromospheric fibrils.
4. **Linear Programming (LP) Venn-Region Signed Distance Field (SDF) Boundary Calibration (`src.postprocess`)**: Resolves multi-annotator boundary disagreement on 2-pass and 3-pass GONG observations via exact 8-region Venn decomposition and Euclidean distance transform level sets while strictly enforcing disjoint 2D instance masks.
5. **Full Competition Contract & Geometric Audit (`src.evaluation`)**: Verifies 100% test stem coverage (180/180 observations), zero 2D pixel overlap collisions, and station-stratified instance statistics.

## 1. Environment Initialization & Modular Package Imports

In [1]:
import os
import sys
import time
import cv2
import numpy as np
import pandas as pd
import torch

# Import modular Apex Sovereign source package
from src import (
    detect_solar_disk,
    limb_darkening_correction,
    normalize_solar_disk,
    PositiveBiasedSolarDataset,
    SoftDiceLoss,
    AsymmetricTverskyLoss,
    LovaszHingeLoss,
    ApexCompoundLoss,
    build_segmentation_model,
    predict_d4_tta,
    logit_space_ensemble,
    topological_grouping_decoupled,
    extract_component_features,
    GBDTComponentFilter,
    VennSDFBoundaryCalibrator,
    rle_encode,
    rle_decode,
    compute_panoptic_quality,
    verify_submission_contract,
)
from reproduce_submission import ApexSovereignPipeline

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | OpenCV: {cv2.__version__} | NumPy: {np.__version__}")

Python: 3.11.9 | PyTorch: 2.6.0+cu124 | OpenCV: 5.0.0 | NumPy: 2.4.6


## 2. Solar Disk Detection, Limb Darkening Correction & Normalization
Full-disk GONG H-$\alpha$ filtergrams ($2048 \times 2048$) exhibit radial intensity roll-off toward the solar limb:
$$I_{\text{corr}}(r) = \frac{I(r)}{1 - u_1(1 - \mu) - u_2(1 - \mu)^2}, \qquad \mu = \sqrt{1 - (r/R_{\odot})^2}$$
Below we verify disk center/radius estimation and intensity normalization on a synthetic full-disk solar observation.

In [2]:
H = W = 2048
yy, xx = np.ogrid[:H, :W]
r2 = (xx - 1024.0) ** 2 + (yy - 1024.0) ** 2
R_sun = 950.0
on_disk = r2 <= (R_sun ** 2)

# Simulate physical limb darkening + dark curvilinear H-alpha filaments
mu_sim = np.sqrt(np.clip(1.0 - r2 / (R_sun ** 2), 0.0, 1.0))
limb_profile = (1.0 - 0.62 * (1.0 - mu_sim) + 0.08 * ((1.0 - mu_sim) ** 2)) * on_disk
synth_obs = (limb_profile * 185.0).astype(np.uint8)

# Inject a curvilinear filament with a 4-pixel intensity bridge dip
cv2.line(synth_obs, (720, 910), (860, 955), 42, thickness=12)
cv2.line(synth_obs, (865, 957), (1010, 1005), 42, thickness=12)
# Inject a small circular sunspot pore (false-alarm distractor)
cv2.circle(synth_obs, (1250, 1180), 9, 50, thickness=-1)

cx, cy, radius = detect_solar_disk(synth_obs)
img_norm, disk_stats = normalize_solar_disk(synth_obs, apply_ldc=True)
print(f"Detected Solar Disk: center=({cx:.1f}, {cy:.1f}), radius={radius:.1f} px (True R={R_sun})")
print(f"Normalized Disk Stats: mean={disk_stats['mean']:.2f}, std={disk_stats['std']:.2f}, tensor_shape={img_norm.shape}")

Detected Solar Disk: center=(1024.0, 1024.0), radius=948.3 px (True R=950.0)
Normalized Disk Stats: mean=184.43, std=5.60, tensor_shape=(2048, 2048)


## 3. Multi-Backbone Losses & Logit-Space Ensembling
To prevent narrow curvilinear filaments ($<0.8\%$ of disk pixels) from vanishing under standard cross-entropy, we combine Soft Dice, Asymmetric Tversky ($\alpha=0.3, \beta=0.7$), and Lovász-Softmax hinge losses, and ensemble posterior probability maps in uncompressed logit space:
$$P_{\text{grand}}(x, y) = \sigma\!\left(\sum_{m=1}^{M} w_m \,\text{logit}\big(P_m(x, y)\big)\right)$$

In [3]:
# Verify compound loss functions on synthetic logits and targets
dummy_logits = torch.randn(2, 1, 128, 128, requires_grad=True)
dummy_targets = (torch.rand(2, 1, 128, 128) > 0.95).float()

loss_dice = ApexCompoundLoss(mode="dice")(dummy_logits, dummy_targets)
loss_tversky = ApexCompoundLoss(mode="tversky", alpha=0.3, beta=0.7)(dummy_logits, dummy_targets)
loss_lovasz = ApexCompoundLoss(mode="lovasz")(dummy_logits, dummy_targets)
print(f"Compound Losses -> Dice+BCE: {loss_dice.item():.4f} | Tversky+BCE: {loss_tversky.item():.4f} | Lovasz+BCE: {loss_lovasz.item():.4f}")

# Demonstrate 4-family logit-space ensembling
base_prob = 1.0 / (1.0 + np.exp(-4.0 * (np.clip(-img_norm, 0.0, 6.0) / 3.2 - 0.55)))
p_grand = logit_space_ensemble({
    "resnet34_dice": np.clip(base_prob * 0.98, 1e-4, 1 - 1e-4),
    "effnet_b3_dice": np.clip(base_prob * 1.01, 1e-4, 1 - 1e-4),
    "effnet_b3_tversky": np.clip(base_prob ** 0.90, 1e-4, 1 - 1e-4),
    "densenet121_lovasz": np.clip(base_prob * 1.00, 1e-4, 1 - 1e-4),
})
print(f"Logit-Space Grand Ensemble range: [{p_grand.min():.4f}, {p_grand.max():.4f}]")

Compound Losses -> Dice+BCE: 0.8546 | Tversky+BCE: 0.8316 | Lovasz+BCE: 1.6810
Logit-Space Grand Ensemble range: [0.1069, 0.9983]


## 4. Decoupled Topological Spine Grouping ($R=6$) & 18-Feature GBDT Component Filter
Standard connected components (`R=0`) split the filament at the 4-pixel absorption dip into two fragments that both fail $\text{IoU} > 0.50$ against the continuous ground-truth spine. Decoupled $R=6$ topological grouping unifies the spine while pruning the circular sunspot pore distractor.

In [4]:
# Compare R=0 (naive connected components) vs R=6 (decoupled spine grouping)
raw_r0 = topological_grouping_decoupled(p_grand, t_group=0.40, t_trim=0.480, group_radius=0, min_area=150)
raw_r6 = topological_grouping_decoupled(p_grand, t_group=0.40, t_trim=0.480, group_radius=6, min_area=150)

gbdt_filter = GBDTComponentFilter(tau=0.38)
filtered_r6 = gbdt_filter.filter_instances(raw_r6, p_grand, img_norm)

print(f"Candidates at R=0 (fragmented): {len(raw_r0)} components")
print(f"Candidates at R=6 (unified spine + pore): {len(raw_r6)} components")
print(f"Final Kept Filaments after 18-Feature GBDT Filter: {len(filtered_r6)} true filament(s)")
feats_0 = extract_component_features(filtered_r6[0], p_grand, img_norm)
print(f"Kept Filament Area={feats_0[0]:.0f} px, Circularity={feats_0[10]:.3f}, Eccentricity={feats_0[15]:.3f}")

Candidates at R=0 (fragmented): 2 components
Candidates at R=6 (unified spine + pore): 2 components
Final Kept Filaments after 18-Feature GBDT Filter: 1 true filament(s)
Kept Filament Area=4084 px, Circularity=0.105, Eccentricity=0.999


## 5. Venn-Region Signed Distance Field (SDF) Multi-Annotator Boundary Calibration
On multi-pass observations (where 2 or 3 astrophysicists independently traced the same solar disk with $\approx 63\%$ pairwise agreement), our 8-region Venn Linear Programming (LP) solver constructs a single non-overlapping medial mask that simultaneously exceeds $\text{IoU} > 0.50$ across all passes while maximizing $\sum_{k} \text{IoU}_k$.

In [5]:
# Simulate 3 discordant human annotator passes on a filament
gt_core = np.zeros((H, W), dtype=bool)
gt_core[900:935, 740:960] = True
pass_1 = gt_core.copy()
pass_2 = np.roll(gt_core, 4, axis=0)
pass_3 = np.roll(gt_core, -3, axis=1)
avail_canvas = np.ones((H, W), dtype=bool)

lp_solution = VennSDFBoundaryCalibrator.solve_venn_3pass_lp(
    pass_1, pass_2, pass_3, avail_canvas, u_cap=0.8490, t_min=0.5015
)
print("3-Pass Venn-SDF LP Solution:")
print(f"  Per-Pass IoUs : {[round(x, 4) for x in lp_solution['ious']]}")
print(f"  Total Sum IoU : {lp_solution['sum_iou']:.4f} (All 3 passes > 0.50 TP threshold!)")

3-Pass Venn-SDF LP Solution:
  Per-Pass IoUs : [0.8485, 0.8485, 0.8485]
  Total Sum IoU : 2.5455 (All 3 passes > 0.50 TP threshold!)


## 6. Final Submission Verification & Geometric Audit (`submission.csv`)
We now execute the complete structural, geometric, and non-overlapping 2D RLE mask audit on the winning submission file `submission.csv`.

In [6]:
t0 = time.time()
sub_path = "submission.csv" if os.path.exists("submission.csv") else "../submission.csv"
df_sub = pd.read_csv(sub_path)

audit = verify_submission_contract(df_sub, expected_stems=180, check_overlaps=True)
print("=" * 68)
print("APEX SOVEREIGN FINAL SUBMISSION AUDIT (GLOBAL RANK 1 - 0.64 PQ)")
print("=" * 68)
print(f"Submission File          : {os.path.abspath(sub_path)}")
print(f"Total Filament Instances : {audit['num_rows']:,}")
print(f"Unique Test Stems Covered: {audit['num_stems']} / 180 (100.0% Coverage)")
print(f"2D Mask Overlap Count    : {audit['overlap_violations']} (Strictly Disjoint)")
print(f"Mean Filament Area       : {audit['mean_area_px']:.1f} px (Median: {audit['median_area_px']:.1f} px)")
print(f"GONG Station Distribution: {audit['station_counts']}")
print(f"Audit Completed in       : {time.time() - t0:.2f}s")
print("=" * 68)
df_sub.head(10)

APEX SOVEREIGN FINAL SUBMISSION AUDIT (GLOBAL RANK 1 - 0.64 PQ)
Submission File          : C:\project\kaggle\Solar Filament Segmentation Challenge 2026\final_submission_package\code\submission.csv
Total Filament Instances : 1,413
Unique Test Stems Covered: 180 / 180 (100.0% Coverage)
2D Mask Overlap Count    : 0 (Strictly Disjoint)
Mean Filament Area       : 1847.1 px (Median: 1086.0 px)
GONG Station Distribution: {'B': 271, 'C': 290, 'L': 259, 'M': 186, 'T': 211, 'U': 196}
Audit Completed in       : 43.41s


,filament_id,segmentation_rle
0,20110120105534Ch_1,Z_Zh21no15L6J4MO100N2O022M0O10O11N2O00002NO2N100^PNIK0\o17dPN1\o1:0O1O1O0000O2M31\OhPNb0Yo1N3OO1010O000O2N1O1O10O01O1O0100O10O1OO201O1N10O10O1O1K4000009G20O0O1N4L6J3N1O00O2M3O11N2N1HbPNL_o15`PNJao16`PNHbo173100O001O1000000000O01O1O2O0O1OJbPNK^o14cPNJ^o16cPNH^o18501O000O10001O000O10000001O0000O3NW^]P1
1,20110120105534Ch_2,Skhn13eo19N1O1O1O0010M30O01000LDcPN<\o1GcPN8]o160N101O100N2M210000000OO2O2N1O10O1000O1O10O01O10O01O0O1N3OO110OO100O1O1O1O0ON1021H[OWQNd0en1BYQN>gn1DVQN92]Ohn1o0311N10O100N101O1O10O0IKUOeQNk0[n1[O[QNi0fn180000000J\QNQOdn1o0]QNoNdn1R15O01O010N101NZOXQN7hn1IYQN7fn1CYQNM1?gn1DYQNJ2a0fn1DaQN;_n1DbQN;an1AaQN`0on101O10O01CgPN1fo1O0O010O001O001O1OdWcj1
2,20110130110334Ch_1,Q[PP2749Qo1GoPN;nn1ERQN=mn1;O1O2N3M2N1N2K5L120001O1000000DaQNWO_n1h0cQNWO]n1h0dQNXO\n1g0fQNXOZn1h0fQNXOZn11aQN=5BYn1OgQN=0CZn1NhQN?MD[n1KjQNa0KCjn1;WQNFin18XQNIgn17ZQNIfn16[QNKcn13`QNM_n10UQNG>8^n11dQNN]n11dQNN^n1OeQNNWcim1
3,20110130110334Ch_2,a_hd252Lfo19M200N3M2O6J0UQN\OZn1e0cQN]O^n1b0`QN@an1?^QNBdn1=ZQNDgn1<WQNEjn1;TQNFmn1e011O00001fQNoNcm16kQNb00[O:Njm15mQNi08TOim12PRNk05UOjm10QRNl06SOhm11SRNP11oNkm11URNQ1OmNlm13TRNP10mNmm11URNR1NlNmm12VRNR1KmNom11WRNR1ImNPn11XRNR1FmNSn11XRNV1hm1jNYRNU1gm1kNZRNT1fm1lN[RNS1fm1kN[RNV14lNgl1NVSN1Cl07UOI2Um1KbSNn0^OVOJ3Sn1g0RRNWOI3Vn1g0nQNXOL1Vn1g0lQNXO00Un1h0iQNXO30Tn1R1lQNmNVn1S1iQNnNVn1S1hQNnNXn1R1hQNoN]n1X12fNcQNl0]n1ROcQNo0hn1O000000O10O101O0O2O2N5K1O0000001ZOkPN?Uo1_OnPN`0Zo1O2N1B_PN<bo1100O010O0010O0001O01O010O1O3N0O1O1O0010O000010O0001N100O20O001O1O1O1O1O0001O001O6J3N0O10N2O1N001O01O101N10001O1O1O10O101N2N1O001O00O1K5L4N201OGmPNFQo1:PQNFQo17RQNHon14TQNLmn1OWQN1]o1O00001O00e\kQ1
4,20110214175414Mh_1,U]j[27ho12IHePN8Yo1;K3N3N1nNZOgRNg0Xm1ZOhRNg0Vm1[OhRNf0Xm1ZOhRNf0Xm1[OgRNf0Xm1[OgRNe0Xm1]OgRN1\O3mm1MfRNNA4hm1NgRNMD4dm10gRNKH3am13fRNJK1_m16dRNJOO\m18cRNJ2NZm19ZRNA09=LYm1;WRN3a0BWm1S1jRNlNVm1T1kRNlNTm1T1lRNmNSm1S1oRNkNQm1U1PSNiNQm1W1PSNgNQm1Y1l0UOaQN2an1JdQN3^n1DlQN:on1M]cSb1
5,20110214175414Mh_2,fYeg2=bo12N2N1O1L2O2O01000N5M010L4O2O0OO0N4O010O0M4J5O2O1O0N3O001O1N12O0O0100O010O1000O01O100O100O100O100000001O000O10001O01O00000000000000O01000000000000000001O010O000010O01O10O01O010O0100O010O001O01O01O000001O00001O0O1O1000O01000O100O101N1OWgnP1
6,20110214175414Mh_3,`^jd22mo12O1O00001O00001O000MJ\PN7bo1J]PN7ao142O001O002N1O0000MI]PN7co1I\PN8do1201O02N2N010O00M302N11OO010000O1O0100001O00O001000000O011O0O0002O0O1OGJhPN6Wo1MgPN3Yo1NePN3[o191O1O1O2NL401O0002N5K1OLEbPN=\o14001_OePN;bo1N2N1O001O0100O0010O01O010O1O002O0O102MK`PNI`o1751O1O3M2OO0O2K410O0011N01O2N2N001O[`mS1
7,20110329082654Uh_1,dYam11ko15N2O0O2O0O2M2O001O10O1O1O001O010O010000O01O1O10O010O100O11O2M1O100O10O10OO2O001O010OO2O1000O01000O100O1O0]PNGao18aPNH]o19aPNGbo1<O2N0O101O001O00O100001O1O000000O100O0100O100000N111O0FcPN1_o1MaPN3\o1McPNN07[o1:0000O110N02O000010M^OlPN1K7Xo1JPQN8mn1ISQN7mn1JRQN6mn1LSQN3mn1MSQN2nn10nPN1So10kPN1Uo10jPN0Vo1=001O03NO1O1O1O100O001O1O2N1O1O010O1O00001N2O0O100O2O0O2O00000O10XPNKeo14[PNMeo12\PNNco13\PNLfo13ZPNNeo17O1O1O1O001O0100O01O1N101O000000000000O10N1100OLCgPN?Xo14000000O0ZO^OQRN`0Sn1EcQN=`n1EXQN?jn1:2N2N1N2N3J8L2I:J^hnf1
8,20110329082654Uh_10,_b`Z12mo1101O1O1O1O01O010O0001O002N01O1O1O0100O010O1000O1O100O01O100O4MO1O1O10O01001XPNIco1:1O1N10O101O00002N3NO1M3O2M1020N100000O1N11O0010O01O0001O0001O01O0001O01N102M2O\\a`2
9,20110329082654Uh_11,QXmb21lo14O1O1N3OO01O01O100O1O01O000O2O001O1N100010N2000O1N2O0O1O1O1O010O01O0001O000OO12OO10M30000001M3O001O0000000O100N20010O001O10O101O1O00OO02O00000O10100O100O100O02ON110O2O000O1O1O1O100OWiZW1
